# LLM-as-a-Judge

For anything open-ended — a summary, an explanation, a piece of code with no test suite —
there is no string to compare against, so the scorer has to be a model. That works
surprisingly well and fails in surprisingly *systematic* ways.

The distinction that matters: a judge's errors are **not random noise**. A noisy judge
costs you statistical power, which more samples can fix. A *biased* judge costs you
validity, and no amount of sampling fixes it — you will confidently measure the wrong
thing. This notebook is mostly about finding and quantifying the biases.

Companion to [Eval Harness Design](eval-harness-design.ipynb) (the judge is one scorer
inside a harness) and [The Statistics of Evals](eval-statistics.ipynb).

## 1. What & Why

Two shapes, and the choice matters more than the prompt:

- **Pointwise** — show the judge one response, ask for a score (1–5, or pass/fail
  against a rubric). Cheap, `O(n)`, and directly interpretable. But absolute scores from
  a model are poorly calibrated and drift between judge versions.
- **Pairwise** — show two responses, ask which is better. Much more reliable, because
  relative judgements are easier than absolute ones. Costs `O(n²)` in principle, and
  needs an aggregation model (Elo, Bradley–Terry) to turn comparisons into rankings.

**Reach for a judge when** the output is open-ended, you need to evaluate faster or
cheaper than humans can, and you are comparing *systems* rather than certifying absolute
quality.

**Don't when** a programmatic check exists. If the task has unit tests, an exact answer,
or a schema, use those — they are cheaper, deterministic, and unbiasable. A judge is what
you use when you have nothing better, not a default.

**The rule that makes this scientific:** a judge is itself an instrument that must be
**validated against human labels** on a sample. Without that, you have a number with
unknown validity. Report judge-human agreement the way you would report an inter-annotator
agreement statistic.

## 2. Mental Model

**A reviewer with predictable prejudices.**

A judge model is a competent reviewer who has never been told they have habits:

- They tend to prefer whichever response they read **first** (or, depending on the model,
  the last) — **position bias**.
- They mistake length for thoroughness — **verbosity bias**.
- They rate text in their own style more highly — **self-preference bias**.
- They are reluctant to use the bottom of a scale, so a 1–5 rubric behaves like a 3–5
  rubric — **scale compression**.

Every one of these is measurable with the same trick: **present the same content in a
different arrangement and see whether the verdict changes.** A judge that gives
inconsistent verdicts on logically identical inputs cannot be measuring quality.

That trick is the whole methodology. Swap the order → position bias. Pad one answer with
harmless text → verbosity bias. Compare against human labels → validity.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Pointwise / pairwise** | Score one response, versus compare two. Pairwise is more reliable; pointwise is cheaper and gives absolute numbers. |
| **Position bias** | Preference for the response in a particular slot, independent of content. Measured by swapping. |
| **Consistency rate** | Fraction of pairs where the judge gives the same verdict under both orderings. The judge's own reliability ceiling. |
| **Verbosity bias** | Correlation between response length and judged quality, beyond what quality justifies. |
| **Self-preference bias** | A judge scoring its own family's outputs higher. |
| **Rubric** | Explicit criteria the judge scores against. The single biggest quality lever, and it must include what *not* to reward. |
| **Reference-guided judging** | Giving the judge a gold answer to compare against. Substantially raises agreement where a reference exists. |
| **Cohen's κ** | Chance-corrected agreement. Raw agreement is misleading when one verdict dominates. |
| **Bradley–Terry / Elo** | Turning pairwise comparisons into a single latent quality score per system. |
| **Ties** | Allowing "about equal" changes the aggregation and usually improves agreement — forced choice injects noise. |
| **Judge ensembling** | Several judges, or several orderings, voted. The cheapest large reliability gain available. |

## 4. Setup

NumPy only. Each example simulates a judge with a specific, known bias so the *detection
method* can be validated — you can see it recover a bias whose true size you set.

In [1]:
# %pip install numpy

import numpy as np
from math import comb  # noqa: F401  (used in the exercises)

rng = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — measuring position bias by swapping

Simulate a judge with a genuine (if imperfect) sense of quality *plus* a thumb on the
scale for whichever response comes first. Then recover that bias knowing only the
verdicts.

In [2]:
def simulate_judge(quality_a, quality_b, position_bias=0.0, noise=0.35, seed=0):
    '''Verdict for one pair. Returns 'A' or 'B'.

    The judge compares noisy perceptions of quality, plus a fixed bonus to whichever
    response is shown FIRST (slot A).
    '''
    r = np.random.default_rng(seed)
    perceived_a = quality_a + position_bias + r.normal(0, noise)
    perceived_b = quality_b + r.normal(0, noise)
    return "A" if perceived_a > perceived_b else "B"

n_pairs = 4000
qual_x = rng.normal(0, 1, n_pairs)      # system X's quality on each item
qual_y = rng.normal(0, 1, n_pairs)      # system Y's

TRUE_BIAS = 0.5

for bias in (0.0, TRUE_BIAS):
    # Present X first, then present Y first -- the SAME content, swapped.
    v1 = [simulate_judge(qual_x[i], qual_y[i], bias, seed=i) for i in range(n_pairs)]
    v2 = [simulate_judge(qual_y[i], qual_x[i], bias, seed=i) for i in range(n_pairs)]

    # In ordering 2, a verdict of 'A' means Y won.
    x_wins_1 = np.array([v == "A" for v in v1])
    x_wins_2 = np.array([v == "B" for v in v2])

    consistent = float(np.mean(x_wins_1 == x_wins_2))
    first_slot_wins = float(np.mean([v == "A" for v in v1 + v2]))
    print(f"position bias = {bias:.2f}")
    print(f"  X win-rate, X shown first : {x_wins_1.mean():.1%}")
    print(f"  X win-rate, X shown second: {x_wins_2.mean():.1%}")
    print(f"  first-slot win-rate       : {first_slot_wins:.1%}   (50% == unbiased)")
    print(f"  consistency under swap    : {consistent:.1%}")
    print()

print("The detection method needs no ground truth: run every pair BOTH ways.")
print("A first-slot win-rate away from 50% is position bias, full stop -- averaged over")
print("both orderings the content is identical, so nothing else can produce it.")
print("\nThe fix is the same as the measurement: judge both orderings and average.")
print("A pair where the two orderings disagree is a genuine tie, and treating it as one")
print("is more honest than taking whichever verdict you happened to ask for first.")

position bias = 0.00
  X win-rate, X shown first : 49.5%
  X win-rate, X shown second: 47.5%
  first-slot win-rate       : 51.0%   (50% == unbiased)
  consistency under swap    : 78.6%

position bias = 0.50
  X win-rate, X shown first : 62.7%
  X win-rate, X shown second: 35.8%
  first-slot win-rate       : 63.5%   (50% == unbiased)
  consistency under swap    : 68.9%

The detection method needs no ground truth: run every pair BOTH ways.
A first-slot win-rate away from 50% is position bias, full stop -- averaged over
both orderings the content is identical, so nothing else can produce it.

The fix is the same as the measurement: judge both orderings and average.
A pair where the two orderings disagree is a genuine tie, and treating it as one
is more honest than taking whichever verdict you happened to ask for first.


### Example 2 — raw agreement lies; use a chance-corrected statistic

You validate your judge against human labels and get 85% agreement. Whether that is good
depends entirely on the label distribution.

In [3]:
def cohens_kappa(a, b):
    '''Chance-corrected agreement between two label sequences.'''
    a, b = np.asarray(a), np.asarray(b)
    po = float(np.mean(a == b))
    labels = set(a.tolist()) | set(b.tolist())
    pe = sum((np.mean(a == l)) * (np.mean(b == l)) for l in labels)
    return po, (po - pe) / (1 - pe) if pe < 1 else 0.0

scenarios = {
    "balanced labels (50/50)": 0.50,
    "skewed labels (85/15)": 0.85,
    "very skewed (95/5)": 0.95,
}
for name, p_major in scenarios.items():
    human = rng.random(4000) < p_major
    # A judge that simply always predicts the majority class -- zero real skill.
    lazy = np.ones(4000, dtype=bool)
    po, k = cohens_kappa(human, lazy)
    print(f"{name:26} always-majority judge: raw agreement {po:.0%}, kappa {k:+.3f}")

print("\nA judge with NO skill at all reaches 95% raw agreement when the labels are 95/5.")
print("Kappa correctly reports ~0. Always report a chance-corrected statistic.\n")

# What a judge with real skill looks like, for comparison.
human = rng.random(4000) < 0.5
for skill in (0.60, 0.75, 0.90):
    judge = np.where(rng.random(4000) < skill, human, ~human)
    po, k = cohens_kappa(human, judge)
    print(f"judge with {skill:.0%} true skill: raw {po:.0%}, kappa {k:.3f}")
print("\nRule of thumb: kappa below ~0.4 means the judge is not measuring what the")
print("humans are measuring, and no amount of extra sampling will fix that.")

balanced labels (50/50)    always-majority judge: raw agreement 50%, kappa +0.000
skewed labels (85/15)      always-majority judge: raw agreement 85%, kappa +0.000
very skewed (95/5)         always-majority judge: raw agreement 96%, kappa +0.000

A judge with NO skill at all reaches 95% raw agreement when the labels are 95/5.
Kappa correctly reports ~0. Always report a chance-corrected statistic.

judge with 60% true skill: raw 60%, kappa 0.202
judge with 75% true skill: raw 75%, kappa 0.505
judge with 90% true skill: raw 90%, kappa 0.797

Rule of thumb: kappa below ~0.4 means the judge is not measuring what the
humans are measuring, and no amount of extra sampling will fix that.


### Example 3 — verbosity bias, and why it is hard to see

A judge that partly rewards length. The trap: longer answers often *are* better, so a
raw length-score correlation proves nothing. You need an intervention — pad a response
with content that adds no information and see if the score moves.

In [4]:
n = 3000
true_quality = rng.normal(0, 1, n)
# Longer answers genuinely tend to be a bit better -- the confound.
length = 200 + 60 * true_quality + rng.normal(0, 50, n)

LENGTH_WEIGHT = 0.004      # the judge's hidden thumb on the scale, per character

judged = true_quality + LENGTH_WEIGHT * (length - length.mean()) + rng.normal(0, 0.3, n)

def corr(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(np.corrcoef(a, b)[0, 1])

print("OBSERVATIONAL (what you can compute from logs):")
print(f"  corr(length, judge score)   = {corr(length, judged):+.3f}")
print(f"  corr(length, true quality)  = {corr(length, true_quality):+.3f}")
print("  -> the judge correlates with length. But so does real quality, so this")
print("     alone does NOT demonstrate bias.\n")

print("INTERVENTIONAL (the experiment that settles it):")
# Take the same responses and pad them with filler that adds no quality.
PAD = 150
judged_padded = (true_quality + LENGTH_WEIGHT * (length + PAD - length.mean())
                 + rng.normal(0, 0.3, n))
delta = float(np.mean(judged_padded - judged))
print(f"  add {PAD} characters of contentless padding to every response")
print(f"  mean change in judged score = {delta:+.3f}")
print(f"  (true quality changed by 0.000 by construction)")
print(f"\n  win-rate of the padded version against its own unpadded self: "
      f"{np.mean(judged_padded > judged):.1%}")
print("\nA fair judge would sit at 50%. Padding cannot improve a response, so anything")
print("above 50% is bias measured directly, with the quality confound held fixed.")
print("\nThis is the general recipe: to test for bias in feature F, change F while")
print("holding quality constant, and see whether the verdict moves.")

OBSERVATIONAL (what you can compute from logs):
  corr(length, judge score)   = +0.837
  corr(length, true quality)  = +0.771
  -> the judge correlates with length. But so does real quality, so this
     alone does NOT demonstrate bias.

INTERVENTIONAL (the experiment that settles it):
  add 150 characters of contentless padding to every response
  mean change in judged score = +0.598
  (true quality changed by 0.000 by construction)

  win-rate of the padded version against its own unpadded self: 92.4%

A fair judge would sit at 50%. Padding cannot improve a response, so anything
above 50% is bias measured directly, with the quality confound held fixed.

This is the general recipe: to test for bias in feature F, change F while
holding quality constant, and see whether the verdict moves.


### Example 4 — turning pairwise verdicts into a ranking (Bradley–Terry)

Pairwise comparisons are more reliable than absolute scores, but you need to aggregate
them. Bradley–Terry fits one latent strength per system such that
`P(i beats j) = σ(sᵢ − sⱼ)`.

In [5]:
systems = ["baseline", "sft", "rlhf-v1", "rlhf-v2"]
true_strength = np.array([0.0, 0.8, 1.4, 1.6])

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

# Simulate a round-robin tournament of pairwise judgements.
K = len(systems)
wins = np.zeros((K, K))
GAMES = 300
for i in range(K):
    for j in range(K):
        if i >= j:
            continue
        p = sigmoid(true_strength[i] - true_strength[j])
        w = rng.binomial(GAMES, p)
        wins[i, j], wins[j, i] = w, GAMES - w

def bradley_terry(wins, iters=500, lr=0.1):
    '''Fit latent strengths by gradient ascent on the log-likelihood.'''
    K = wins.shape[0]
    s = np.zeros(K)
    for _ in range(iters):
        grad = np.zeros(K)
        for i in range(K):
            for j in range(K):
                if i == j:
                    continue
                n = wins[i, j] + wins[j, i]
                if n == 0:
                    continue
                grad[i] += wins[i, j] - n * sigmoid(s[i] - s[j])
        s += lr * grad / wins.sum()
        s -= s[0]                      # fix the gauge: baseline == 0
    return s

fitted = bradley_terry(wins)
print(f"{'system':12} {'raw win-rate':>13} {'BT strength':>12} {'true':>8}")
for k, name in enumerate(systems):
    played = wins[k].sum() + wins[:, k].sum()
    raw = wins[k].sum() / played if played else 0
    print(f"{name:12} {raw:13.1%} {fitted[k]:12.2f} {true_strength[k]:8.2f}")

print("\nBradley-Terry recovers the latent strengths up to an additive constant.")
print("\nThis tournament is a BALANCED round-robin, so raw win-rate happens to rank")
print("correctly too -- everyone faced the same opponents. Note the numbers are still")
print("not interchangeable: win-rate is compressed into [0,1] and is not linear in")
print("strength, so it cannot tell you how much better rlhf-v2 is than sft.")
print("\nBT earns its place when pairings are UNBALANCED, which is the normal case for")
print("arena-style data: a system that mostly faced weak opponents has an inflated")
print("win-rate, and BT corrects for opponent strength. It also gives you a likelihood,")
print("hence confidence intervals on the ranking.")

system        raw win-rate  BT strength     true
baseline             24.0%         0.00     0.00
sft                  45.4%         0.74     0.80
rlhf-v1              61.1%         1.25     1.40
rlhf-v2              69.4%         1.54     1.60

Bradley-Terry recovers the latent strengths up to an additive constant.

This tournament is a BALANCED round-robin, so raw win-rate happens to rank
correctly too -- everyone faced the same opponents. Note the numbers are still
not interchangeable: win-rate is compressed into [0,1] and is not linear in
strength, so it cannot tell you how much better rlhf-v2 is than sft.

BT earns its place when pairings are UNBALANCED, which is the normal case for
arena-style data: a system that mostly faced weak opponents has an inflated
win-rate, and BT corrects for opponent strength. It also gives you a likelihood,
hence confidence intervals on the ranking.


## 6. Gotchas & Pitfalls

- **Never validating against humans.** Without an agreement statistic on a sample, the
  judge's numbers have unknown validity. This is the single most common omission.
- **Reporting raw agreement.** Example 2: a no-skill judge hits 95% agreement on skewed
  labels. Use κ or another chance-corrected measure.
- **Judging one ordering only.** Example 1. Run both and average; disagreement is a tie.
- **Concluding bias from a correlation.** Example 3: longer answers really are often
  better. Bias claims need an intervention that holds quality fixed.
- **Using the model family you are evaluating as its own judge.** Self-preference is
  well documented. Use a different family, or at least report that you did not.
- **Treating judge scores as an interval scale.** The gap between 3 and 4 is not the gap
  between 4 and 5, and judges compress the bottom of the scale. Prefer pairwise, or
  treat pointwise scores as ordinal.
- **Changing the judge model mid-experiment.** Judge versions are not comparable. Pin the
  version, and re-run the baseline whenever you change it.
- **A rubric that only says what to reward.** Judges reward fluency and length by
  default; the rubric has to say *not* to. Explicitly instructing "do not reward length
  or confident tone" measurably reduces those biases.
- **Forcing a choice with no tie option.** When two responses are genuinely equivalent, a
  forced choice is a coin flip recorded as a signal.
- **Letting the judge see which system produced which response.** Any label — a name, a
  formatting quirk, a system prompt — reintroduces every bias you controlled for.

## 7. When to Use vs Alternatives

| Situation | Reach for |
|---|---|
| A programmatic check exists (tests, exact answer, schema) | **That check.** Cheaper, deterministic, unbiasable |
| Open-ended quality, comparing two systems | **Pairwise judging**, both orderings, ties allowed |
| Open-ended quality, absolute threshold needed | **Pointwise with a rubric** — and calibrate against human labels |
| Ranking many systems from sparse comparisons | **Bradley–Terry / Elo** (Example 4) |
| A reference answer exists | **Reference-guided judging** — a large, cheap accuracy gain |
| The highest-stakes decisions | **Humans**, on a sample, with the judge used to triage |

**The honest position.** A judge is a *cheap approximation to human evaluation*, and its
value is entirely determined by how well it approximates on your data. That is knowable —
it is one labelling exercise on a few hundred items — and it converts the judge from an
unaccountable oracle into an instrument with known error characteristics.

The most reliable configuration available cheaply: pairwise, both orderings averaged, ties
allowed, a rubric that names what not to reward, a judge from a different model family
than the systems under test, and a human-agreement number reported alongside every result.

## 8. Resources

- [Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://arxiv.org/abs/2306.05685) — the foundational study; quantifies position, verbosity and self-enhancement bias, and reports judge-human agreement.
- [Large Language Models are not Fair Evaluators](https://arxiv.org/abs/2305.17926) — position bias in depth, and the calibration strategies that reduce it.
- [Length-Controlled AlpacaEval](https://arxiv.org/abs/2404.04475) — a concrete statistical correction for verbosity bias in a live leaderboard.
- [G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment](https://arxiv.org/abs/2303.16634) — chain-of-thought rubric scoring and its agreement characteristics.
- [Chatbot Arena: An Open Platform for Evaluating LLMs by Human Preference](https://arxiv.org/abs/2403.04132) — the Bradley–Terry machinery of Example 4 at scale, including confidence intervals on rankings.
- [Cohen's kappa](https://en.wikipedia.org/wiki/Cohen%27s_kappa) — the statistic from Example 2, with its known weaknesses under extreme skew.

You need to score open-ended responses and a programmatic check is available for the task. Why prefer the programmatic check over a judge model?

The notebook describes a judge as 'a reviewer with predictable prejudices'. Describe the single experimental trick that detects every one of those prejudices, and apply it to both position bias and verbosity bias.

You validate a judge against human labels and get 92% agreement. Why might this be worthless?

In [ ]:
def first_slot_win_rate(verdicts_a_first, verdicts_b_first):
    ...


In [ ]:
assert abs(first_slot_win_rate(['A','B'], ['A','B']) - 0.5) < 1e-9, 'unbiased'
assert abs(first_slot_win_rate(['A','A'], ['A','A']) - 1.0) < 1e-9, 'always first slot'
assert abs(first_slot_win_rate(['B','B'], ['B','B']) - 0.0) < 1e-9
assert abs(first_slot_win_rate(['A','A','A'], ['B']) - 0.75) < 1e-9
assert first_slot_win_rate([], []) == 0.0, 'must not divide by zero'


Which practice most undermines the validity of a pairwise judging setup?

Describe the most reliable pairwise judging configuration available cheaply, listing at least four specific choices, and say what must be reported alongside the result.